
# SILVER LAYER - FHIR Observation Processing (Production Ready)



In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
import logging
import json
from pyspark.sql.functions import (
    col, trim, lower, row_number
)
from pyspark.sql.window import Window

In [0]:
# ====================== Config Details ======================
CONFIG = {
    "resource": "Observation",
    "run_date": datetime.today().strftime("%Y-%m-%d"),
    "bronze_path": "fhir_data.prd_bronze",
    "silver_path": "fhir_data.prd_silver"
}

resource = CONFIG["resource"]
run_date = CONFIG["run_date"]
bronze_base = CONFIG["bronze_path"]
silver_base = CONFIG["silver_path"]

In [0]:
# ====================== LOGGING ======================
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

logger.info(f"Starting Silver Layer | Resource: {resource} | Run Date: {run_date}")

In [0]:
# ====================== IMPORTS ======================
from pyspark.sql.types import *
from pyspark.sql.functions import *

# ====================== MAIN FUNCTION ======================
def process_patient_silver(resource, run_date, bronze_base):

    bronze_path = f"{bronze_base}.{resource.lower()}"

    df = spark.read\
        .table(bronze_path) \
        .filter(col("load_date") == run_date) \
        .dropDuplicates()

    # ====================== SCHEMA ======================
    schema = StructType([
        StructField("id", StringType()),
        StructField("resourceType", StringType()),
        StructField("status", StringType()),

        StructField("subject", StructType([
            StructField("reference", StringType())
        ])),

        StructField("encounter", StructType([
            StructField("reference", StringType())
        ])),

        StructField("effectiveDateTime", StringType()),
        StructField("effectivePeriod", StructType([
            StructField("start", StringType()),
            StructField("end", StringType())
        ])),

        StructField("code", StructType([
            StructField("text", StringType()),
            StructField("coding", ArrayType(
                StructType([
                    StructField("system", StringType()),
                    StructField("code", StringType()),
                    StructField("display", StringType())
                ])
            ))
        ])),

        StructField("valueQuantity", StructType([
            StructField("value", DoubleType()),
            StructField("unit", StringType()),
            StructField("system", StringType())
        ])),

        StructField("valueInteger", IntegerType()),

        StructField("meta", StructType([
            StructField("lastUpdated", StringType()),
            StructField("versionId", StringType())
        ]))
    ])

    df_parsed = df.withColumn("parsed", from_json(col("resource"), schema))

    # ====================== FLATTEN ======================
    df_flat = (
        df_parsed
        .withColumn("coding", explode_outer(col("parsed.code.coding")))
        .withColumn(
            "observation_value",
            coalesce(
                col("parsed.valueQuantity.value"),
                col("parsed.valueInteger").cast("double")
            )
        )
        .withColumn(
            "effective_ts",
            coalesce(
                to_timestamp(col("parsed.effectiveDateTime")),
                to_timestamp(col("parsed.effectivePeriod.start"))
            )
        )
    )

    df_silver = df_flat.select(
        col("parsed.id").alias("observation_id"),
        split(col("parsed.subject.reference"), "/")[1].alias("patient_id"),
        split(col("parsed.encounter.reference"), "/")[1].alias("encounter_id"),

        col("coding.system").alias("code_system"),
        col("coding.code").alias("code"),
        col("coding.display").alias("code_display"),
        col("parsed.code.text").alias("code_text"),

        col("observation_value"),
        col("parsed.valueQuantity.unit").alias("unit"),
        col("parsed.valueQuantity.system").alias("unit_system"),

        col("parsed.status").alias("status"),
        col("effective_ts"),
        to_timestamp(col("parsed.meta.lastUpdated")).alias("last_updated_ts"),
        col("parsed.meta.versionId").alias("version_id"),
        current_date().alias("ingestion_date")
    )

    # ====================== HARD SILVER FILTERS ======================
    df_silver = (
        df_silver
        .filter(col("status") == "final")
        .filter(col("observation_value").isNotNull())
        .filter(col("code").isNotNull())
        .filter(col("code") != "85354-9")  # DROP BP PANEL
    )

    # ====================== OBSERVATION TYPE ======================
    df_silver = df_silver.withColumn(
        "observation_type",
        when(col("code").isin("8867-4", "9279-1", "8310-5", "8480-6", "8462-4"), "VITAL_SIGN")
        .when(col("code").isin("55423-8", "93832-4", "82305-4"), "WEARABLE")
        .when(col("code").isin("718-7", "2093-3", "2132-9", "6690-2", "789-8", "778-1"), "LAB")
        .when(col("code").isin("LP248386-6", "59574-4", "93858-9"), "ANTHROPOMETRIC")
    )

    # ====================== TEXT NORMALIZATION ======================
    df_silver = df_silver.withColumn(
        "code_text_clean",
        coalesce(col("code_text"), col("code_display"))
    )

    # ====================== CANONICAL UNITS ======================
    df_silver = df_silver.withColumn(
        "unit_clean",
        when(col("code") == "8867-4", "beats/min")
        .when(col("code") == "9279-1", "breaths/min")
        .when(col("code").isin("8480-6", "8462-4"), "mmHg")
        .when(col("code") == "55423-8", "steps")
        .when(col("code") == "6690-2", "10^3/uL")
        .when(col("code") == "789-8", "10^6/uL")
        .when(col("code") == "778-1", "10^3/uL")
        .otherwise(col("unit"))
    )

    df_silver = df_silver.withColumn(
        "unit_system_clean",
        when(col("unit_clean").isNotNull(), "http://unitsofmeasure.org")
    )

    # ====================== DOMAIN VALIDATION ======================
    df_silver = df_silver.withColumn(
        "is_valid",
        when((col("code") == "55423-8") & col("observation_value").between(100, 50000), True)      # Steps
        .when((col("code") == "93832-4") & (col("observation_value") >= 60), True)                 # Sleep (minutes)
        .when((col("code") == "8867-4") & col("observation_value").between(30, 220), True)         # Heart rate
        .when(col("code").isin("8480-6", "8462-4") & col("observation_value").between(40, 250), True)
        .when(col("observation_type") == "LAB", True)
        .when(col("observation_type") == "ANTHROPOMETRIC", True)
        .otherwise(False)
    )

    # ====================== FINAL SILVER ======================
    df_silver_final = (
        df_silver
        .filter(col("is_valid") == True)
        .withColumn(
            "data_source_type",
            when(col("observation_type").isin("LAB", "VITAL_SIGN"), "CLINICAL")
            .otherwise("DEVICE")
        )
        .select(
            "observation_id",
            "patient_id",
            "encounter_id",
            "observation_type",
            "data_source_type",
            "code",
            "code_text_clean",
            "observation_value",
            "unit_clean",
            "unit_system_clean",
            "status",
            "effective_ts",
            "last_updated_ts",
            "version_id",
            "ingestion_date"
        )
    )
    
#     df_silver_final = df_silver_final.filter(df_silverservation_value") < 500))
# )((col("observation_type") == "WEARABLE") &


    df_silver_final.display()
    return df_silver_final



process_patient_silver(resource, run_date, bronze_base)

In [0]:

# ====================== Main EXECUTION ======================
try:
    df_silver = process_patient_silver(resource, run_date, bronze_base)
    df_silver.createOrReplaceTempView('silver_obse')

    silver_path = f"{silver_base}.{resource.lower()}"
    spark.sql(f"""
              MERGE INTO {silver_path} t
USING silver_obse s
ON t.observation_id = s.observation_id 
AND t.is_current = true

-- 1. EXPIRE OLD RECORD
WHEN MATCHED AND (
    t.patient_id <> s.patient_id OR
    t.encounter_id <> s.encounter_id OR
    t.observation_type <> s.observation_type OR
    t.data_source_type <> s.data_source_type OR
    t.code <> s.code OR
    t.code_text_clean <> s.code_text_clean OR
    t.observation_value <> s.observation_value OR
    t.unit_clean <> s.unit_clean OR
    t.unit_system_clean <> s.unit_system_clean OR
    t.status <> s.status OR
    t.effective_ts <> s.effective_ts OR
    t.last_updated_ts <> s.last_updated_ts OR
    t.version_id <> s.version_id
)

THEN UPDATE SET
    t.is_current = false,
    t.effective_end_date = current_timestamp()

-- 2. INSERT NEW RECORD
WHEN NOT MATCHED
THEN INSERT (
    observation_id,
    patient_id,
    encounter_id,
    observation_type,
    data_source_type,
    code,
    code_text_clean,
    observation_value,
    unit_clean,
    unit_system_clean,
    status,
    effective_ts,
    last_updated_ts,
    version_id,
    ingestion_date,
    effective_start_date,
    effective_end_date,
    is_current
)
VALUES (
    s.observation_id,
    s.patient_id,
    s.encounter_id,
    s.observation_type,
    s.data_source_type,
    s.code,
    s.code_text_clean,
    s.observation_value,
    s.unit_clean,
    s.unit_system_clean,
    s.status,
    s.effective_ts,
    s.last_updated_ts,
    s.version_id,
    s.ingestion_date,
    current_timestamp(),
    NULL,
    true
)
              
              """)

    # df_silver.write.format("delta") \
    #     .mode("append") \
    #     .option("mergeSchema", "true") \
    #     .saveAsTable(silver_path)

    logger.info(f"Successfully written to Silver: {silver_path}")

    # df_silver.display()
    # df_silver.printSchema()

except Exception as e:
    logger.error(f"Pipeline failed: {str(e)}", exc_info=True)
    raise

finally:
    logger.info("Pipeline execution completed")